In [0]:
import pyspark.sql.functions as F

T_Trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_cc_member = spark.table ("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_emails = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_film = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_ticket = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")
T_sales = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_sales_channel")

In [0]:
latest_snapshot = T_cc_member.agg(F.max("snapshot_date_id")).first()[0]

T_Trans = (
    T_Trans
    .filter(F.col("VisitDateId") > "20210101")
    .filter(F.col("TicketFLAG") == 'Yes')
)


from pyspark.sql import Window

w_email = Window.partitionBy("CDE_ID").orderBy(F.col("CDE_ID"))  # replace with real ordering

T_emails_pivot = (
    T_emails
    .withColumn("Email_Address", F.lower(F.col("Email_Address")))
    .withColumn("_rn", F.row_number().over(w_email))
    .groupBy("CDE_ID")
    .agg(
        F.max(F.when(F.col("_rn") == 1, F.col("Email_Address"))).alias("Primary_Email"),
        F.max(F.when(F.col("_rn") == 2, F.col("Email_Address"))).alias("Alternate_Email"),
    )
    .withColumnRenamed("CDE_ID", "cde_id")
)

T_member_details = (
    T_cc_member
    .filter(F.col("snapshot_date_id") == latest_snapshot)
    .select(
        "cde_id", "cc_user_profile_id", "plan_id", "renewal_status",
        "recognitions_redeemed", "recognitions_left",
        "current_term_start_datetime", "current_term_end_datetime", "snapshot_date_id"
    )
    .join(T_emails_pivot, on="cde_id", how="inner")
    .withColumn(
        "plan_id",
        F.when(F.col("plan_id") == "monthly1", "Monthly")
        .when(F.col("plan_id") == "annual1", "Annual")
        .when(F.col("plan_id") == "gift-plan", "Gift Plan")
        .otherwise(F.col("plan_id"))
    )
    .withColumn(
        "current_term_start_datetime",
        F.to_date(F.col("current_term_start_datetime"))
    )

    .withColumn(
        "current_term_end_datetime",
        F.to_date(F.col("current_term_end_datetime"))
    )
    .withColumn(
        "snapshot_date_id",
        F.to_date(
            F.col("snapshot_date_id"),
            "yyyyMMdd"
        )
    )
    .withColumn(
    "Search_Column",
    F.concat_ws(" ", F.col("Primary_Email"), F.coalesce(F.col("Alternate_Email"), F.lit(""))))
    .filter(F.col("recognitions_redeemed") > 0)
)

In [0]:
(
    T_member_details.write
    .mode("overwrite")
    .option("overwriteSchema", "true")  # only if schema may have changed
    .saveAsTable("data_exploration_hub.marketing_hub.GS_CC_Members")
)

In [0]:
T_member_transactions = (
    T_member_details.select("cde_id").distinct()
    .join(T_Trans.withColumnRenamed("CDE_ID","cde_id"),  on="cde_id", how="left")
    .join(T_items.select("itemskey", "Item_ID", "Item_Description"), T_Trans.ItemSkey == T_items.itemskey, how="inner")
    .join(T_film.select("film_id", "title_name"), T_Trans.film_id == T_film.film_id, how="inner")
    .join(T_location, T_Trans.LocationId == T_location.LocationID, "inner")
    .join(T_ticket, T_Trans.TicketTypeSkey == T_ticket.tickettypeskey, how="inner")
    .join(T_sales,T_Trans.SalesChannelSkey == T_sales.saleschannelskey, how= "inner")
    .withColumn(
        "Discount_Type",
        F.when(F.col("CineClub_Discount_Type_ID") == 0, "Non CineClub member")
         .when(F.col("CineClub_Discount_Type_ID") == 1, "CineClub member no discount")
         .when(F.col("CineClub_Discount_Type_ID") == 2, "CineClub member 20% on concession")
         .when(F.col("CineClub_Discount_Type_ID") == 3, "CineClub member 20% on games")
         .when(F.col("CineClub_Discount_Type_ID") == 4, "CineClub member 10% on movie rentals and purchases")
         .when(F.col("CineClub_Discount_Type_ID") == 5, "CineClub member $9 ticket")
         .when(F.col("CineClub_Discount_Type_ID") == 6, "CineClub member free ticket")
    )
    .withColumn(
        "VisitDateId",
        F.to_date(
            F.col("VisitDateId"),
            "yyyyMMdd"
        )
    )
    .select(
        "cde_id",
        "VisitDateId",
        "TransactionDateid",
        "Transaction_Number",
        "sales_channel",
        "title_name",
        "Item_Description",
        "ticket_type_description_english",
        "Discount_Type",
        "Quantity",
        "SCENE_Points_Earned",
        "SCENE_Points_Redeemed",
        "Net_Spend",
        "Net_Revenue",
        "AdvanceFLAG",
        "TicketFLAG",
        "ConcessionFLAG",
        "Location_Name",
        "Province_CD",
        )
)

In [0]:
(
    T_member_transactions.write
    .mode("overwrite")
    .option("overwriteSchema", "true")  # only if schema may have changed
    .saveAsTable("data_exploration_hub.marketing_hub.GS_CC_Transactions")
)

In [0]:
T_member_transactions.display()